# Panel interactivo de ventas de vehículos ligeros en México

## Objetivo general

Desarrollar una aplicación web interactiva que permita explorar las
ventas de vehículos ligeros reportadas al INEGI en México durante
2021–2026, mediante filtros y visualizaciones por marca, modelo,
segmento y periodo.

El panel estará orientado a responsables comerciales y de logística
interesados en identificar tendencias de ventas, comparar competidores
y obtener información que apoye su planificación comercial.

El análisis considerará los años completos de 2021–2025 y los datos
disponibles de 2026 hasta septiembre, utilizando periodos equivalentes
para las comparaciones de crecimiento.

## Preparación de los datos

Se revisará la descripción de los vehículos para separar sus
características en columnas y convertir las ventas a valores numéricos,
conservando los datos ausentes.

In [1]:
import pandas as pd

In [2]:
ventas = pd.read_csv('../ventas_autos_mexico.csv')
ventas.head(10)

,anio,mes,descripcion_vehiculo,ventas
0,2021,Enero,Acura - ADX - Camiones ligeros - SUV's - Nacio...,-
1,2021,Enero,Acura - DEM - Automóviles - Compactos - Import...,-
2,2021,Enero,Acura - DEMO - Automóviles - Compactos - Impor...,-
3,2021,Enero,Acura - ILX - Automóviles - De Lujo - Importad...,2
4,2021,Enero,Acura - INTEGRA - Automóviles - De Lujo - Impo...,-
5,2021,Enero,Acura - Mdx - Camiones ligeros - SUV's - Impor...,-
6,2021,Enero,Acura - Mdx - Camiones ligeros - SUV's - Impor...,11
7,2021,Enero,Acura - NSX - Automóviles - Deportivos - Impor...,0
8,2021,Enero,Acura - Rdx - Camiones ligeros - SUV's - Impor...,42
9,2021,Enero,Acura - Rl - Automóviles - De Lujo - Importado...,-


In [3]:
ventas.info()

<class 'pandas.DataFrame'>
RangeIndex: 106920 entries, 0 to 106919
Data columns (total 4 columns):
 #   Column                Non-Null Count   Dtype
---  ------                --------------   -----
 0   anio                  106920 non-null  int64
 1   mes                   106920 non-null  str  
 2   descripcion_vehiculo  106920 non-null  str  
 3   ventas                106920 non-null  str  
dtypes: int64(1), str(3)
memory usage: 11.2 MB


In [4]:
#Normalización del texto de la columna "descripción_vehículo" para evitar errores de tipeo y mayúsculas/minúsculas
ventas['descripcion_vehiculo'] = (
    ventas['descripcion_vehiculo']
    .str.strip()
    .str.lower()
)

ventas.head(10)

,anio,mes,descripcion_vehiculo,ventas
0,2021,Enero,acura - adx - camiones ligeros - suv's - nacio...,-
1,2021,Enero,acura - dem - automóviles - compactos - import...,-
2,2021,Enero,acura - demo - automóviles - compactos - impor...,-
3,2021,Enero,acura - ilx - automóviles - de lujo - importad...,2
4,2021,Enero,acura - integra - automóviles - de lujo - impo...,-
5,2021,Enero,acura - mdx - camiones ligeros - suv's - impor...,-
6,2021,Enero,acura - mdx - camiones ligeros - suv's - impor...,11
7,2021,Enero,acura - nsx - automóviles - deportivos - impor...,0
8,2021,Enero,acura - rdx - camiones ligeros - suv's - impor...,42
9,2021,Enero,acura - rl - automóviles - de lujo - importado...,-


In [5]:
# Número de valores únicos en cada columna
ventas.nunique()

anio                       6
mes                       12
descripcion_vehiculo    1455
ventas                  2236
dtype: int64

In [6]:
# Verificación de valores nulos en cada columna
ventas.isna().sum()

anio                    0
mes                     0
descripcion_vehiculo    0
ventas                  0
dtype: int64

In [7]:
# Verificación de espacios en blanco al inicio o al final de los valores de la columna "descripcion_vehiculo"
descripcion = ventas['descripcion_vehiculo']

con_espacios = (descripcion != descripcion.str.strip()) & descripcion.notna()

con_espacios.sum()

np.int64(0)

In [8]:
# Separar desde la derecha las últimas cuatro características
partes = ventas['descripcion_vehiculo'].str.rsplit(
    ' - ', n=4, expand=True
)

# Separar marca y modelo
marca_modelo = partes[0].str.split(' - ', n=1, expand=True)

# Unir las partes y asignar nombres únicos
separadas = pd.concat(
    [marca_modelo, partes.iloc[:, 1:]],
    axis=1
)

columnas = ['marca', 'modelo', 'tipo', 'segmento', 'origen', 'pais_origen']
separadas.columns = columnas

ventas[columnas] = separadas

ventas.head()

,anio,mes,descripcion_vehiculo,ventas,marca,modelo,tipo,segmento,origen,pais_origen
0,2021,Enero,acura - adx - camiones ligeros - suv's - nacio...,-,acura,adx,camiones ligeros,suv's,nacional,méxico
1,2021,Enero,acura - dem - automóviles - compactos - import...,-,acura,dem,automóviles,compactos,importado,canadá
2,2021,Enero,acura - demo - automóviles - compactos - impor...,-,acura,demo,automóviles,compactos,importado,canadá
3,2021,Enero,acura - ilx - automóviles - de lujo - importad...,2,acura,ilx,automóviles,de lujo,importado,estados unidos
4,2021,Enero,acura - integra - automóviles - de lujo - impo...,-,acura,integra,automóviles,de lujo,importado,estados unidos


In [9]:
# Convertir la columna "ventas" a tipo numérico, reemplazando los valores "-" por NaN
ventas["ventas"]= pd.to_numeric(
    ventas["ventas"].replace("-", pd.NA),
    errors="raise"
).astype("Int64")

ventas.info()

<class 'pandas.DataFrame'>
RangeIndex: 106920 entries, 0 to 106919
Data columns (total 10 columns):
 #   Column                Non-Null Count   Dtype
---  ------                --------------   -----
 0   anio                  106920 non-null  int64
 1   mes                   106920 non-null  str  
 2   descripcion_vehiculo  106920 non-null  str  
 3   ventas                34031 non-null   Int64
 4   marca                 106920 non-null  str  
 5   modelo                106920 non-null  str  
 6   tipo                  106920 non-null  str  
 7   segmento              106920 non-null  str  
 8   origen                106920 non-null  str  
 9   pais_origen           106920 non-null  str  
dtypes: Int64(1), int64(1), str(8)
memory usage: 21.8 MB


## Exploración y visualización de los datos


In [10]:
# Agrupar por año y calcular la cobertura de datos
cobertura_anual = ventas.groupby('anio')['ventas'].agg(
    registros_totales='size',
    registros_con_datos='count'
)

cobertura_anual['registros_ausentes'] = (
    cobertura_anual['registros_totales']
    - cobertura_anual['registros_con_datos']
)

cobertura_anual

,registros_totales,registros_con_datos,registros_ausentes
anio,,,
2021,17820,5099,12721
2022,17820,5119,12701
2023,17820,5478,12342
2024,17820,6280,11540
2025,17820,6876,10944
2026,17820,5179,12641


In [11]:
# Calcular las ventas totales por año, considerando los valores ausentes
ventas_anuales = ventas.groupby('anio')['ventas'].sum(min_count=1).reset_index(name='ventas_totales')
ventas_anuales.head()

,anio,ventas_totales
0,2021,1014735
1,2022,1094728
2,2023,1363714
3,2024,1504322
4,2025,1524635


In [12]:
# Calcular las ventas totales por año y marca, considerando los valores ausentes
ventas_marca_anio = ventas.groupby(['anio', 'marca'])['ventas'].sum(min_count=1).reset_index(name='ventas_totales')

ventas_marca_anio.head(10)

,anio,marca,ventas_totales
0,2021,acura,971
1,2021,alfa romeo,142
2,2021,audi,9416
3,2021,auteco,<NA>
4,2021,bentley,19
5,2021,bmw,12895
6,2021,bmw group,<NA>
7,2021,changan,<NA>
8,2021,chirey,<NA>
9,2021,chrysler,47164


In [16]:
# Visualización de las ventas por marca en México: 2021–2026 (2026 hasta septiembre)
import plotly.express as px

marcas_elegidas = ['nissan', 'toyota', 'volkswagen', 'honda', 'chevrolet', 'kia', 'mazda', 'ford', 'bmw']

datos_grafico = ventas_marca_anio.loc[
    (ventas_marca_anio['anio'] <= 2026)
    & ventas_marca_anio['marca'].isin(marcas_elegidas)
].dropna(subset=['ventas_totales'])

fig = px.line(
    datos_grafico,
    x='anio',
    y='ventas_totales',
    color='marca',
    title='Ventas por marca en México: 2021–2026 (2026 hasta septiembre)',
    labels={'anio': 'Año', 'ventas_totales': 'Vehículos Vendidos', 'marca': 'Marca'},
)

fig.update_xaxes(dtick=1)

fig.show()

In [18]:
# Definir el orden de los meses
orden_meses = [
    'enero', 'febrero', 'marzo', 'abril', 'mayo', 'junio',
    'julio', 'agosto', 'septiembre', 'octubre', 'noviembre', 'diciembre'
]

ventas['mes_numero'] = ventas['mes'].str.strip().str.lower().map(
    {mes: numero for numero, mes in enumerate(orden_meses, start=1)}
)

ventas[['mes', 'mes_numero']].drop_duplicates().sort_values('mes_numero')

,mes,mes_numero
0,Enero,1
1485,Febrero,2
2970,Marzo,3
4455,Abril,4
5940,Mayo,5
7425,Junio,6
8910,Julio,7
10395,Agosto,8
11880,Septiembre,9
13365,Octubre,10


In [ ]:
# Calcular las ventas totales por año y mes, considerando los valores ausentes
ventas_mensuales = (
    ventas.groupby(['anio', 'mes_numero', 'mes'])['ventas']
    .sum(min_count=1)
    .reset_index(name='ventas_totales')
)

ventas_mensuales.head(10)

,anio,mes_numero,mes,ventas_totales
0,2021,1,Enero,81657
1,2021,2,Febrero,82863
2,2021,3,Marzo,96319
3,2021,4,Abril,84287
4,2021,5,Mayo,86710
5,2021,6,Junio,88688
6,2021,7,Julio,82157
7,2021,8,Agosto,78235
8,2021,9,Septiembre,76930
9,2021,10,Octubre,76640


In [ ]:
# Calcular las ventas totales por año y mes, considerando los valores ausentes
ventas_mensuales_marca = (
    ventas.groupby(['anio', 'mes_numero', 'marca'])['ventas']
    .sum(min_count=1)
    .reset_index(name='ventas_totales')
)
ventas_mensuales_marca.head(10)

,anio,mes_numero,marca,ventas_totales
0,2021,1,acura,64
1,2021,1,alfa romeo,14
2,2021,1,audi,627
3,2021,1,auteco,<NA>
4,2021,1,bentley,1
5,2021,1,bmw,985
6,2021,1,bmw group,<NA>
7,2021,1,changan,<NA>
8,2021,1,chirey,<NA>
9,2021,1,chrysler,3326


In [ ]:
# Visualización de las ventas mensuales de una marca específica en México: 2025 y 2026
marca_elegida = 'toyota'

datos_mensuales = ventas_mensuales_marca.loc[
    (ventas_mensuales_marca['marca'] == marca_elegida)
    & ventas_mensuales_marca['anio'].isin([2025, 2026])
].copy()

# Usar el año como categoría para obtener una línea por año
datos_mensuales['anio'] = datos_mensuales['anio'].astype(str)

datos_mensuales = datos_mensuales.sort_values(['anio', 'mes_numero'])

fig = px.line(
    datos_mensuales,
    x='mes_numero',
    y='ventas_totales',
    color='anio',
    markers=True,
    title=f'Ventas mensuales de {marca_elegida.title()}: 2025 y 2026',
    labels={
        'mes_numero': 'Mes',
        'ventas_totales': 'Vehículos vendidos',
        'anio': 'Año'
    }
)

fig.update_xaxes(
    tickmode='array',
    tickvals=list(range(1, 13)),
    ticktext=[mes.capitalize() for mes in orden_meses]
)

fig.show()


In [28]:
# Visualización de la distribución de ventas mensuales de una marca específica en México: 2025


datos_histograma = ventas_mensuales_marca.loc[
    (ventas_mensuales_marca['marca'] == marca_elegida)
    & (ventas_mensuales_marca['anio'] == 2025)
].dropna(subset=['ventas_totales'])

fig_histograma = px.histogram(
    datos_histograma,
    x='ventas_totales',
    nbins=6,
    title=f'Distribución de ventas mensuales de {marca_elegida.title()} en 2025',
    labels={'ventas_totales': 'Vehículos vendidos en un mes'}
)

fig_histograma.update_yaxes(title_text='Cantidad de meses')
fig_histograma.show()

In [ ]:
# Comparación de ventas mensuales de una marca específica en México: 2025 vs 2026 (hasta septiembre)
datos_histograma.sort_values('mes_numero')[
    ['mes_numero', 'ventas_totales']
]

,mes_numero,ventas_totales
2544,1,10976
2596,2,8232
2648,3,10894
2700,4,9022
2752,5,10974
2804,6,10162
2856,7,10471
2908,8,9755
2960,9,9893
3012,10,11977


In [ ]:
# Comparación de ventas mensuales de una marca específica en México: 2025 vs 2026 (hasta septiembre)
comparacion_mensual = (
    ventas_mensuales_marca.loc[
        (ventas_mensuales_marca['marca'] == marca_elegida)
        & ventas_mensuales_marca['anio'].isin([2025, 2026])
        & (ventas_mensuales_marca['mes_numero'] <= 9)
    ]
    .pivot(
        index='mes_numero',
        columns='anio',
        values='ventas_totales'
    )
    .rename(columns={2025: 'ventas_2025', 2026: 'ventas_2026'})
    .dropna(subset=['ventas_2025', 'ventas_2026'])
    .reset_index()
)

comparacion_mensual

anio,mes_numero,ventas_2025,ventas_2026
0,1,10976,11669
1,2,8232,8116
2,3,10894,10129
3,4,9022,10713
4,5,10974,10840
5,6,10162,10662
6,7,10471,11074
7,8,9755,10870
8,9,9893,10205


In [32]:
# Visualización de la dispersión de ventas mensuales de una marca específica en México: 2025 vs 2026 (hasta septiembre)
fig_dispersion = px.scatter(
    comparacion_mensual,
    x='ventas_2025',
    y='ventas_2026',
    text='mes_numero',
    hover_data=['mes_numero'],
    title=f'Ventas de {marca_elegida.title()}: enero–septiembre de 2025 y 2026',
    labels={
        'ventas_2025': 'Vehículos vendidos en 2025',
        'ventas_2026': 'Vehículos vendidos en 2026',
        'mes_numero': 'Mes'
    }
)

fig_dispersion.update_traces(textposition='top center')

# Línea de referencia: mismas ventas en ambos años
limite = comparacion_mensual[
    ['ventas_2025', 'ventas_2026']
].max().max()

fig_dispersion.add_shape(
    type='line',
    x0=0, y0=0,
    x1=limite, y1=limite,
    line=dict(color='gray', dash='dash')
)

fig_dispersion.show()

## Observaciones y próximos pasos

- La comparación anual permite explorar la evolución de ventas por marca.
  El total de 2026 es parcial y no debe compararse directamente con años completos.
- El histograma de Toyota en 2025 muestra que cinco meses registraron
  ventas entre 10,000 y 11,000 vehículos.
- En la comparación mensual de Toyota, seis de los nueve meses analizados
  tuvieron mayores ventas en 2026 que en 2025.
- Los datos ausentes se conservaron como tales, sin convertirlos en cero.

Estas visualizaciones servirán como base para una aplicación en Streamlit
con filtros de marca y periodo, y controles para mostrar el histograma
y el gráfico de dispersión.